# LLM Zoomcamp Homework 5: Monitoring
Using OpenTelemetry and Gemini API.

In [35]:
# Setup imports
import sqlite3
import pandas as pd
from dotenv import load_dotenv
from google import genai

from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from gitsource import GithubRepositoryDataReader
from minsearch import Index
from rag_helper import RAGBase

# Load env variables and prepare documents
load_dotenv('../hw1/.env')
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp',
    commit_id='8c1834d',
    allowed_extensions={'md'},
    filename_filter=lambda path: '/lessons/' in path,
)
documents = [file.parse() for file in reader.read()]

# Create minsearch index
index = Index(text_fields=['content'], keyword_fields=['filename'])
index.fit(documents)


## Initialize OpenTelemetry Tracer

In [36]:
provider = TracerProvider()
tracer = provider.get_tracer('llm-zoomcamp')


## RAG Pipeline with OpenTelemetry Spans

In [37]:
class RAGTraced(RAGBase):
    def search(self, query, num_results=5):
        with tracer.start_as_current_span('search'):
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span('llm') as span:
            response = super().llm(prompt)
            
            # Extract usage metrics from Gemini response
            input_tokens = response.usage_metadata.prompt_token_count
            output_tokens = response.usage_metadata.candidates_token_count
            cost = (input_tokens / 1000) * 0.000075 + (output_tokens / 1000) * 0.0003
            
            # Log metrics as span attributes
            span.set_attribute('input_tokens', input_tokens)
            span.set_attribute('output_tokens', output_tokens)
            span.set_attribute('cost', cost)
            
            return response

    def rag(self, query):
        with tracer.start_as_current_span('rag'):
            return super().rag(query)


## Answers for Q1, Q2, Q3 (In-Memory Tracing)

In [38]:
# We use InMemorySpanExporter to capture spans directly into a list without printing messy JSONs
memory_exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(memory_exporter))

# Run the RAG pipeline once
client = genai.Client()
traced_rag = RAGTraced(index=index, llm_client=client)

query = 'How does the agentic loop keep calling the model until it stops?'
answer = traced_rag.rag(query)

# Fetch captured spans
spans = memory_exporter.get_finished_spans()
llm_span = next(s for s in spans if s.name == 'llm')

# Calculate answers
q1_spans_count = len(spans)
q2_input_tokens = llm_span.attributes.get('input_tokens')
q3_duration_ms = (llm_span.end_time - llm_span.start_time) / 1_000_000

print(f"Q1 Answer (Number of spans): {q1_spans_count}")
print(f"Q2 Answer (Input tokens): {q2_input_tokens}")
print(f"Q3 Answer (LLM duration): {q3_duration_ms:.2f} ms")


Q1 Answer (Number of spans): 3
Q2 Answer (Input tokens): 7935
Q3 Answer (LLM duration): 1861.54 ms


## Q4 & Q5: Saving traces to SQLite

In [39]:
class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path='traces.db'):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.execute('DELETE FROM spans')
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                'INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)',
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get('input_tokens'),
                    attrs.get('output_tokens'),
                    attrs.get('cost'),
                )
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS
    
    def shutdown(self):
        self.conn.close()


In [40]:
# Add SQLite exporter to our tracer
sqlite_exporter = SQLiteSpanExporter('traces.db')
provider.add_span_processor(SimpleSpanProcessor(sqlite_exporter))

# Run the RAG pipeline again to log to the database
traced_rag.rag(query)

# Load database into Pandas DataFrame
df_spans = pd.read_sql_query('SELECT * FROM spans', sqlite_exporter.conn)

# Q4: Unique span names
print('Q4 Answer (Span names in DB):', df_spans['name'].unique().tolist())


Q4 Answer (Span names in DB): ['search', 'llm', 'rag']


In [41]:
# Q5: Find the longest span (excluding rag)
df_spans['duration_ms'] = (df_spans['end_time'] - df_spans['start_time']) / 1_000_000
df_filtered = df_spans[df_spans['name'] != 'rag']

durations = df_filtered.groupby('name')['duration_ms'].sum()
print(f"Durations (ms):\n{durations.to_string()}")
print(f"\nQ5 Answer (Longest span): {durations.idxmax()}")


Durations (ms):
name
llm       1711.766
search       1.540

Q5 Answer (Longest span): llm


## Q6: Token stability across multiple runs

In [42]:
# Let's run it 3 more times
for _ in range(3):
    traced_rag.rag(query)

# Query all llm spans from the database
df_llm = pd.read_sql_query("SELECT * FROM spans WHERE name = 'llm'", sqlite_exporter.conn)
tokens = df_llm['input_tokens'].tolist()

print(f"Tokens across runs: {tokens}")
print(f"Q6 Answer: {'They are identical' if len(set(tokens)) == 1 else 'They differ'}")


Tokens across runs: [7935, 7935, 7935, 7935]
Q6 Answer: They are identical
